<div class="nb-header">
    <div class="nb-title">Quick Start</div>
    <div class="nb-subtitle">Create your first graph in 5 minutes</div>
    <div class="nb-metadata">
        <span class="nb-duration">5 min</span>
        <span class="nb-difficulty nb-difficulty--beginner">Beginner</span>
    </div>
</div>

## Step 1: Connect to the Platform

The SDK auto-discovers the control plane URL from environment variables.

In [ ]:
from graph_olap import GraphOLAPClient

client = GraphOLAPClient()
print(f"Connected to: {client.base_url}")

## Step 2: Create a Mapping

A mapping defines how warehouse tables become graph nodes and edges.

In [ ]:
mapping = client.mappings.create(
    name="quick-start-example",
    config={
        "nodes": [
            {"label": "Person", "table": "employees", "id_column": "id"}
        ],
        "edges": [
            {"type": "REPORTS_TO", "table": "org_chart", "from": "employee_id", "to": "manager_id"}
        ]
    }
)
print(f"Created mapping: {mapping.name}")

## Step 3: Create a Snapshot

A snapshot builds the graph from your current warehouse data.

In [ ]:
snapshot = client.snapshots.create(mapping_id=mapping.id)
print(f"Created snapshot: {snapshot.id}")

## Step 4: Launch a Graph Instance

An instance is a live, queryable graph database.

In [ ]:
instance = client.instances.create(snapshot_id=snapshot.id)
instance.wait_until_ready()
print(f"Instance ready: {instance.id}")

## Step 5: Query Your Graph

Use Cypher to explore the graph.

In [ ]:
conn = instance.connect()

# Count nodes
result = conn.execute("MATCH (n) RETURN count(n) as node_count")
print(f"Total nodes: {result[0]['node_count']}")

# Find managers with most reports
result = conn.execute("""
    MATCH (p:Person)<-[:REPORTS_TO]-(r:Person)
    RETURN p.name as manager, count(r) as reports
    ORDER BY reports DESC
    LIMIT 5
""")
for row in result:
    print(f"{row['manager']}: {row['reports']} reports")

<div class="nb-navigation">
  <div class="nb-navigation__title">Next Steps</div>
  <div class="nb-card-grid">
    <a href="../02-cypher/01_property_graphs.ipynb" class="nb-card">
      <div class="nb-card__title">Learn Cypher</div>
      <div class="nb-card__description">Understand property graphs and write your first queries</div>
      <div class="nb-card__meta">20 min · Beginner</div>
    </a>
    <a href="../03-algorithms/01_introduction.ipynb" class="nb-card">
      <div class="nb-card__title">Run Algorithms</div>
      <div class="nb-card__description">Introduction to graph algorithms and analytics</div>
      <div class="nb-card__meta">15 min · Beginner</div>
    </a>
    <a href="../01-fundamentals/01_getting_started.ipynb" class="nb-card">
      <div class="nb-card__title">Explore the SDK</div>
      <div class="nb-card__description">Deep dive into the Graph OLAP SDK fundamentals</div>
      <div class="nb-card__meta">15 min · Beginner</div>
    </a>
  </div>
</div>